In [ ]:
from transformers import AutoProcessor, TrainingArguments, Trainer, AutoModel, get_scheduler
from torch.optim import AdamW
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch.nn.functional as F
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from sklearn.metrics import f1_score
import random
from multiprocessing import Pool
import torch.nn as nn
import math
import gc
import timm
from sklearn.model_selection import train_test_split

In [ ]:
init_path = '/kaggle/input/jaguar-re-id/'
train_data = pd.read_csv(init_path + 'train.csv')
test_data = pd.read_csv(init_path + 'test.csv')

unique_labels = train_data['ground_truth'].unique()
n_classes = len(unique_labels)
label2id = dict(zip(unique_labels, range(n_classes)))

IMAGE_SIZE = (448, 448)

In [ ]:
def load_and_resize(args):
    path, name = args
    return name, Image.open(path + name).resize(IMAGE_SIZE).convert('RGB')

train_image_names = train_data['filename']
test_image_names = test_data['query_image'].unique()

with Pool(4) as p:
    train_args = [(init_path + 'train/train/', n) for n in train_image_names]
    train_images = dict(tqdm(p.imap(load_and_resize, train_args), total=len(train_args)))

    test_args = [(init_path + 'test/test/', n) for n in test_image_names]
    test_images = dict(tqdm(p.imap(load_and_resize, test_args), total=len(test_args)))

In [ ]:
class CustomDataset:
    def __init__(self, images, processor, labels=None, is_train=False, aug_funcs=None, aug_prob=0.7):
        self.images = images
        self.labels = labels
        self.processor = processor
        self.is_train = is_train
        self.aug_funcs = aug_funcs
        self.aug_prob = aug_prob

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = self.images[index]
        label = self.labels[index] if self.labels is not None else 0

        if self.is_train and self.aug_prob > random.uniform(0, 1):
            image = random.choice(self.aug_funcs)(image)

        image = self.processor(image, return_tensors='pt')['pixel_values'][0]

        return {
            'pixel_values': torch.tensor(image, dtype=torch.float32),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
from torchvision.transforms.functional import to_tensor, to_pil_image

# потом можно добавить ещё TTA, потому что он хорошо увеличивает метрику
# сделать взвешанный TTA

f1 = transforms.RandomHorizontalFlip(p=1.0)
f2 = transforms.ColorJitter(0.2, 0.2, 0.2)
f3 = transforms.GaussianBlur(5, (0.1, 2.0))
f4 = transforms.RandomRotation(10)
f5 = transforms.RandomErasing(p=1, scale=(0.02, 0.15))
f6 = transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1))

def apply_erasing(img):
    t_img = to_tensor(img)
    erased_img = f5(t_img)
    return to_pil_image(erased_img)

aug_funcs = [
    lambda x: x,
    f1,
    f2,
    f3,
    f4,
    f6,
    lambda x: f1(f2(x)),
    lambda x: f2(f3(x)),
    lambda x: f1(f3(x)),
    lambda x: f1(f2(f3(x))),
    lambda x: f4(f2(x)),
    lambda x: f1(f4(x)),
    lambda x: apply_erasing(x),
    lambda x: apply_erasing(f1(x)),
    lambda x: apply_erasing(f2(x)),
    lambda x: apply_erasing(f3(x)),
    lambda x: apply_erasing(f1(f2(x))),
    lambda x: apply_erasing(f1(f3(x))),
    lambda x: apply_erasing(f4(x)),
    lambda x: apply_erasing(f1(f2(f3(x))))
]

In [ ]:
# aug_funcs = [
#     lambda x: x,
#     transforms.RandomHorizontalFlip(p=1.0)
# ]

In [ ]:
# class ArcFaceHead(nn.Module):
#     def __init__(self, in_features, out_features, s=30.0, m=0.50):
#         super().__init__()
#         self.in_features = in_features
#         self.out_features = out_features
#         self.s = s
#         self.m = m
#         self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
#         nn.init.xavier_uniform_(self.weight)
        
#         self.cos_m = math.cos(m)
#         self.sin_m = math.sin(m)
#         self.th = math.cos(math.pi - m)
#         self.mm = math.sin(math.pi - m) * m

#     def forward(self, input, label):
#         cosine = F.linear(F.normalize(input), F.normalize(self.weight))
#         sine = torch.sqrt((1.0 - torch.pow(cosine, 2)).clamp(0, 1))
#         phi = cosine * self.cos_m - sine * self.sin_m
#         phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        
#         one_hot = torch.zeros(cosine.size(), device=input.device)
#         one_hot.scatter_(1, label.view(-1, 1).long(), 1)
#         output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
#         output *= self.s
#         return output

# class DinoV2ArcFace(nn.Module):
#     def __init__(self, model_name, num_classes):
#         super().__init__()
#         self.backbone = AutoModel.from_pretrained(model_name)
#         self.arcface = ArcFaceHead(self.backbone.config.hidden_size, num_classes)
#         self.loss_fn = nn.CrossEntropyLoss()

#     def forward(self, pixel_values, labels=None):
#         outputs = self.backbone(pixel_values)
#         embeddings = outputs.last_hidden_state[:, 0]
        
#         if labels is not None:
#             logits = self.arcface(embeddings, labels)
#             loss = self.loss_fn(logits, labels)
#             return {"loss": loss, "logits": logits}
        
#         return {"embeddings": F.normalize(embeddings)}

In [ ]:
# def rand_bbox(size, lam):
#     W = size[2]
#     H = size[3]
#     cut_rat = np.sqrt(1. - lam)
#     cut_w = int(W * cut_rat)
#     cut_h = int(H * cut_rat)
#     cx = np.random.randint(W)
#     cy = np.random.randint(H)
#     bbx1 = np.clip(cx - cut_w // 2, 0, W)
#     bby1 = np.clip(cy - cut_h // 2, 0, H)
#     bbx2 = np.clip(cx + cut_w // 2, 0, W)
#     bby2 = np.clip(cy + cut_h // 2, 0, H)
#     return bbx1, bby1, bbx2, bby2

# class ArcFaceHead(nn.Module):
#     def __init__(self, in_features, out_features, s=25.0, m=0.4):
#         super().__init__()
#         self.in_features = in_features
#         self.out_features = out_features
#         self.s = s
#         self.m = m
#         self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
#         nn.init.xavier_uniform_(self.weight)
        
#         self.cos_m = math.cos(m)
#         self.sin_m = math.sin(m)
#         self.th = math.cos(math.pi - m)
#         self.mm = math.sin(math.pi - m) * m

#     def forward(self, input, label):
#         cosine = F.linear(F.normalize(input), F.normalize(self.weight))
#         sine = torch.sqrt((1.0 - torch.pow(cosine, 2)).clamp(0, 1))
#         phi = cosine * self.cos_m - sine * self.sin_m
#         phi = torch.where(cosine > self.th, phi, cosine - self.mm)
        
#         one_hot = torch.zeros(cosine.size(), device=input.device)
#         one_hot.scatter_(1, label.view(-1, 1).long(), 1)
#         output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
#         output *= self.s
#         return output

# class DinoV2ArcFace(nn.Module):
#     def __init__(self, model_name, num_classes):
#         super().__init__()
#         self.backbone = AutoModel.from_pretrained(model_name)
#         self.arcface = ArcFaceHead(self.backbone.config.hidden_size, num_classes)
#         self.loss_fn = nn.CrossEntropyLoss()

#     def forward(self, pixel_values, labels=None):
#         outputs = self.backbone(pixel_values)
#         embeddings = outputs.last_hidden_state[:, 0]
        
#         if labels is not None:
#             logits = self.arcface(embeddings, labels)
#             return {"logits": logits, "loss": self.loss_fn(logits, labels)} 
        
#         return {"embeddings": F.normalize(embeddings)}

# class CutMixArcFaceTrainer(Trainer):
#     def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
#         pixel_values = inputs["pixel_values"]
#         labels = inputs["labels"]

#         if self.is_in_train and np.random.rand() < 0.5:
#             # CutMix
#             lam = np.random.beta(1.0, 1.0)
#             rand_index = torch.randperm(pixel_values.size(0)).to(pixel_values.device)
#             target_a = labels
#             target_b = labels[rand_index]
            
#             bbx1, bby1, bbx2, bby2 = rand_bbox(pixel_values.size(), lam)
#             pixel_values[:, :, bbx1:bbx2, bby1:bby2] = pixel_values[rand_index, :, bbx1:bbx2, bby1:bby2]
#             lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (pixel_values.size(-1) * pixel_values.size(-2)))
            
#             # Forward с labels=target_a (доминантный класс для Margin)
#             outputs = model(pixel_values, labels=target_a)
#             logits = outputs["logits"]
            
#             loss_fn = nn.CrossEntropyLoss()
#             loss = loss_fn(logits, target_a) * lam + loss_fn(logits, target_b) * (1. - lam)
#         else:
#             outputs = model(pixel_values, labels=labels)
#             loss = outputs["loss"]

#         return (loss, outputs) if return_outputs else loss

In [ ]:
# class GeM(nn.Module):
#     def __init__(self, p=3, eps=1e-6):
#         super(GeM, self).__init__()
#         self.p = nn.Parameter(torch.ones(1) * p)
#         self.eps = eps

#     def forward(self, x):
#         return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))).pow(1.0 / self.p)

# class ArcFaceHead(nn.Module):
#     def __init__(self, in_features, out_features, s=30.0, m=0.50):
#         super().__init__()
#         self.in_features = in_features
#         self.out_features = out_features
#         self.s = s
#         self.m = m
#         self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
#         nn.init.xavier_uniform_(self.weight)

#         self.cos_m = math.cos(m)
#         self.sin_m = math.sin(m)
#         self.th = math.cos(math.pi - m)
#         self.mm = math.sin(math.pi - m) * m

#     def forward(self, input, label):
#         cosine = F.linear(F.normalize(input), F.normalize(self.weight))
#         phi = cosine - self.m 
#         phi = torch.where(cosine > self.th, phi, cosine - self.mm)

#         one_hot = torch.zeros(cosine.size(), device=input.device)
#         one_hot.scatter_(1, label.view(-1, 1).long(), 1)
#         output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
#         output *= self.s
#         return output

# class DinoV2ArcFace(nn.Module):
#     def __init__(self, model_name, num_classes, s=30, m=0.5):
#         super().__init__()
#         self.backbone = AutoModel.from_pretrained(model_name)
#         self.feat_dim = self.backbone.config.hidden_size
#         self.gem = GeM()
#         self.bn = nn.LayerNorm(self.feat_dim)
#         self.arcface = ArcFaceHead(self.feat_dim, num_classes, s=s, m=m)
#         self.loss_fn = nn.CrossEntropyLoss()

#     def forward(self, pixel_values, labels=None):
#         outputs = self.backbone(pixel_values)
#         last_hidden_state = outputs.last_hidden_state
        
#         features = last_hidden_state[:, 1:, :]
        
#         B, N, C = features.shape
#         H = W = int(math.sqrt(N))
        
#         if H * W != N:
#             features = features[:, -H*W:, :]

#         features = features.permute(0, 2, 1).reshape(B, C, H, W)
        
#         embeddings = self.gem(features).flatten(1)
#         embeddings = self.bn(embeddings)
        
#         if labels is not None:
#             logits = self.arcface(embeddings, labels)
#             loss = self.loss_fn(logits, labels)
#             return {"logits": logits, "loss": loss}
        
#         return {"embeddings": F.normalize(embeddings)}

# def rand_bbox(size, lam):
#     _, _, H, W = size
#     cut_rat = np.sqrt(1.0 - lam)
#     cut_w = int(W * cut_rat)
#     cut_h = int(H * cut_rat)

#     cx = np.random.randint(W)
#     cy = np.random.randint(H)

#     bbx1 = np.clip(cx - cut_w // 2, 0, W)
#     bby1 = np.clip(cy - cut_h // 2, 0, H)
#     bbx2 = np.clip(cx + cut_w // 2, 0, W)
#     bby2 = np.clip(cy + cut_h // 2, 0, H)
#     return bbx1, bby1, bbx2, bby2

# class AugArcFaceTrainer(Trainer):
#     def __init__(self, *args, 
#                  mixup_prob=0.0, 
#                  cutmix_prob=0.0, 
#                  mixup_alpha=1.0, 
#                  cutmix_alpha=1.0,
#                  backbone_lr=2e-5, 
#                  head_lr=2e-4, 
#                  **kwargs):
#         super().__init__(*args, **kwargs)
#         self.mixup_prob = mixup_prob
#         self.cutmix_prob = cutmix_prob
#         self.mixup_alpha = mixup_alpha
#         self.cutmix_alpha = cutmix_alpha
#         self.backbone_lr = backbone_lr
#         self.head_lr = head_lr
#         self.ce = nn.CrossEntropyLoss()

#     def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
#         pixel_values = inputs["pixel_values"]
#         labels = inputs["labels"]

#         r = np.random.rand()
#         do_mixup = r < self.mixup_prob
#         do_cutmix = (r >= self.mixup_prob) and (r < self.mixup_prob + self.cutmix_prob)

#         if model.training and (do_mixup or do_cutmix):
#             lam = 1.0
#             rand_index = torch.randperm(pixel_values.size(0), device=pixel_values.device)
#             target_a = labels
#             target_b = labels[rand_index]
            
#             pixel_values_aug = pixel_values.clone()

#             if do_mixup:
#                 lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)
#                 pixel_values_aug = lam * pixel_values + (1 - lam) * pixel_values[rand_index]
            
#             elif do_cutmix:
#                 lam = np.random.beta(self.cutmix_alpha, self.cutmix_alpha)
#                 bbx1, bby1, bbx2, bby2 = rand_bbox(pixel_values.size(), lam)
#                 pixel_values_aug[:, :, bby1:bby2, bbx1:bbx2] = pixel_values[rand_index, :, bby1:bby2, bbx1:bbx2]
#                 lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (pixel_values.size(-1) * pixel_values.size(-2)))

#             outputs = model(pixel_values=pixel_values_aug, labels=target_a)
#             logits = outputs["logits"]
            
#             loss = self.ce(logits, target_a) * lam + self.ce(logits, target_b) * (1.0 - lam)
        
#         else:
#             outputs = model(pixel_values=pixel_values, labels=labels)
#             loss = outputs["loss"]

#         return (loss, outputs) if return_outputs else loss

#     def create_optimizer(self):
#         model = self.model
#         no_decay = ("bias", "LayerNorm.weight", "layernorm.weight", "norm.weight", "bn.weight", "bn.bias")
#         wd = self.args.weight_decay

#         head_names = ["arcface", "bn", "gem"]
        
#         def is_head(n):
#             return any(x in n for x in head_names)

#         head_params = []
#         backbone_params = []

#         for n, p in model.named_parameters():
#             if not p.requires_grad:
#                 continue
#             if is_head(n):
#                 head_params.append((n, p))
#             else:
#                 backbone_params.append((n, p))

#         optimizer_grouped_parameters = [
#             {
#                 "params": [p for n, p in backbone_params if not any(nd in n for nd in no_decay)],
#                 "lr": self.backbone_lr,
#                 "weight_decay": wd,
#             },
#             {
#                 "params": [p for n, p in backbone_params if any(nd in n for nd in no_decay)],
#                 "lr": self.backbone_lr,
#                 "weight_decay": 0.0,
#             },
#             {
#                 "params": [p for n, p in head_params if not any(nd in n for nd in no_decay)],
#                 "lr": self.head_lr,
#                 "weight_decay": wd,
#             },
#             {
#                 "params": [p for n, p in head_params if any(nd in n for nd in no_decay)],
#                 "lr": self.head_lr,
#                 "weight_decay": 0.0,
#             },
#         ]

#         self.optimizer = AdamW(
#             optimizer_grouped_parameters,
#             betas=(self.args.adam_beta1, self.args.adam_beta2),
#             eps=self.args.adam_epsilon,
#         )
#         return self.optimizer

In [ ]:
class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6):
        super(GeM, self).__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p), (x.size(-2), x.size(-1))).pow(1.0 / self.p)

class ArcFaceHead(nn.Module):
    def __init__(self, in_features, out_features, s=30.0, m=0.50):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, input, label):
        cosine = F.linear(F.normalize(input), F.normalize(self.weight))
        phi = cosine - self.m
        one_hot = torch.zeros(cosine.size(), device=input.device)
        one_hot.scatter_(1, label.view(-1, 1).long(), 1)
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        return output

class Eva02ReID(nn.Module):
    def __init__(self, num_classes, model_name='eva02_large_patch14_448.mim_m38m_ft_in22k_in1k', s=30.0, m=0.50):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        self.feat_dim = self.backbone.num_features
        self.gem = GeM()
        self.bn = nn.LayerNorm(self.feat_dim) 
        self.arcface = ArcFaceHead(self.feat_dim, num_classes, s=s, m=m)
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, pixel_values, labels=None):
        features = self.backbone.forward_features(pixel_values)
        features = features[:, 1:, :] 
        B, N, C = features.shape
        H = W = int(math.sqrt(N))
        features = features.permute(0, 2, 1).reshape(B, C, H, W)
        embeddings = self.gem(features).flatten(1)
        embeddings = self.bn(embeddings)
        
        if labels is not None:
            logits = self.arcface(embeddings, labels)
            loss = self.loss_fn(logits, labels)
            return {"logits": logits, "loss": loss}
        
        return {"embeddings": F.normalize(embeddings)}

class ReIDTrainer(Trainer):
    def __init__(self, *args, backbone_lr=2e-5, head_lr=2e-4, **kwargs):
        super().__init__(*args, **kwargs)
        self.backbone_lr = backbone_lr
        self.head_lr = head_lr
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        outputs = model(pixel_values=inputs["pixel_values"], labels=inputs["labels"])
        loss = outputs["loss"]
        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self):
        model = self.model
        no_decay = ("bias", "LayerNorm.weight", "layernorm.weight", "norm.weight", "bn.weight", "bn.bias")
        wd = self.args.weight_decay

        head_names = ["arcface", "bn", "gem"]
        
        def is_head(n):
            return any(x in n for x in head_names)

        head_params = []
        backbone_params = []

        for n, p in model.named_parameters():
            if not p.requires_grad:
                continue
            if is_head(n):
                head_params.append((n, p))
            else:
                backbone_params.append((n, p))

        optimizer_grouped_parameters = [
            # Backbone
            {
                "params": [p for n, p in backbone_params if not any(nd in n for nd in no_decay)],
                "lr": self.backbone_lr,
                "weight_decay": wd,
            },
            {
                "params": [p for n, p in backbone_params if any(nd in n for nd in no_decay)],
                "lr": self.backbone_lr,
                "weight_decay": 0.0,
            },
            # Head
            {
                "params": [p for n, p in head_params if not any(nd in n for nd in no_decay)],
                "lr": self.head_lr,
                "weight_decay": wd,
            },
            {
                "params": [p for n, p in head_params if any(nd in n for nd in no_decay)],
                "lr": self.head_lr,
                "weight_decay": 0.0,
            },
        ]

        self.optimizer = AdamW(
            optimizer_grouped_parameters,
            betas=(self.args.adam_beta1, self.args.adam_beta2),
            eps=self.args.adam_epsilon,
        )
        return self.optimizer

In [ ]:
# checkpoint = 'facebook/dinov2-base'
checkpoint = 'timm/eva02_large_patch14_448.mim_m38m_ft_in22k_in1k'
processor = AutoProcessor.from_pretrained(checkpoint)

model = Eva02ReID(num_classes=31, model_name=checkpoint, s=30, m=0.5)

In [ ]:
# train_dataset1 = CustomDataset(
#     images=list(train_images.values()),
#     processor=processor,
#     labels=train_data['ground_truth'].map(label2id),
#     is_train=True,
#     aug_funcs=aug_funcs[:2]
# )

# args1 = TrainingArguments(
#     report_to='none',
#     output_dir='./result',
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=8,
#     per_device_eval_batch_size=8,
#     logging_steps=20,
#     logging_strategy='steps',
#     num_train_epochs=30,
#     warmup_steps=100,
#     lr_scheduler_type='cosine',
#     weight_decay=0.01,
#     fp16=torch.cuda.is_available(),
#     # learning_rate=2e-5
# )
# trainer1 = AugArcFaceTrainer(
#     model=model,
#     args=args1,
#     train_dataset=train_dataset1,
#     mixup_prob=0.6,
#     cutmix_prob=0,
#     backbone_lr=6e-5,
#     head_lr=1e-4
# )
# trainer1.train()

# # train_dataset2 = CustomDataset(
# #     images=list(train_images.values()),
# #     processor=processor,
# #     labels=train_data['ground_truth'].map(label2id),
# #     is_train=True,
# #     aug_funcs=aug_funcs
# # )

# # args2 = TrainingArguments(
# #     report_to='none',
# #     output_dir='./result',
# #     per_device_train_batch_size=2,
# #     gradient_accumulation_steps=8,
# #     per_device_eval_batch_size=8,
# #     logging_steps=20,
# #     logging_strategy='steps',
# #     num_train_epochs=3,
# #     warmup_ratio=0.1,
# #     lr_scheduler_type='cosine',
# #     weight_decay=0.01,
# #     fp16=torch.cuda.is_available(),
# #     # learning_rate=2e-5
# # )
# # trainer2 = AugArcFaceTrainer(
# #     model=model,
# #     args=args2,
# #     train_dataset=train_dataset2,
# #     mixup_prob=0.0,
# #     cutmix_prob=0.0,
# #     backbone_lr=5e-6,
# #     head_lr=3e-5
# # )
# # trainer2.train()

In [ ]:
# train_dataset2 = CustomDataset(
#     images=list(train_images.values()),
#     processor=processor,
#     labels=train_data['ground_truth'].map(label2id),
#     is_train=True,
#     aug_funcs=aug_funcs
# )

# args2 = TrainingArguments(
#     report_to='none',
#     output_dir='./result',
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=8,
#     per_device_eval_batch_size=8,
#     logging_steps=20,
#     logging_strategy='steps',
#     num_train_epochs=3,
#     warmup_ratio=0.1,
#     lr_scheduler_type='cosine',
#     weight_decay=0.05,
#     fp16=torch.cuda.is_available(),
#     # learning_rate=2e-5
# )
# trainer2 = AugArcFaceTrainer(
#     model=model,
#     args=args2,
#     train_dataset=train_dataset2,
#     mixup_prob=0.0,
#     cutmix_prob=0.0,
#     backbone_lr=5e-6,
#     head_lr=3e-5
# )
# trainer2.train()

In [ ]:
images = list(train_images.values())
labels = train_data['ground_truth'].map(label2id).tolist()

t_images, v_images, t_labels, v_labels = train_test_split(
    images, labels, test_size=0.03, random_state=42,
    shuffle=True, stratify=labels
)

val_dataset2 = CustomDataset(
    images=v_images,
    processor=processor,
    labels=v_labels
)

train_dataset2 = CustomDataset(
    images=t_images,
    processor=processor,
    labels=t_labels,
    is_train=True,
    aug_funcs=aug_funcs[:2]
)

args2 = TrainingArguments(
    report_to='none',
    output_dir='./result',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    per_device_eval_batch_size=16,
    logging_steps=20,
    eval_steps=20,
    logging_strategy='steps',
    eval_strategy='steps',
    num_train_epochs=15,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    weight_decay=0.02,
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    # learning_rate=2e-5,
    dataloader_num_workers=4
)
trainer2 = ReIDTrainer(
    model=model,
    args=args2,
    train_dataset=train_dataset2,
    eval_dataset=val_dataset2,
    backbone_lr=2e-5,
    head_lr=1e-4
)
trainer2.train()

In [ ]:
trainer2.save_model("./eva_model")

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
def tta_predict_embeds(model, processor, images, aug_funcs, batch_size):
    all_embeds = []
    for aug_func in tqdm(aug_funcs):
        aug_images = [aug_func(image) for image in images]
        embeds = predict_embeds(model, processor, aug_images, batch_size, leave=False)
        all_embeds.append(embeds)
    all_embeds = np.array(all_embeds).mean(axis=0)
    return all_embeds

def predict_embeds(model, processor, images, batch_size, leave=True):
    all_embeds = []
    for i in tqdm(range(0, len(images), batch_size), leave=leave):
        batch = images[i: i + batch_size]
        batch = processor(batch, return_tensors='pt')
        batch = {k: v.to(next(model.parameters()).device) for k, v in batch.items()}
        with torch.no_grad():
            embeds = model(**batch)['embeddings']
        embeds = embeds.cpu().numpy()
        all_embeds.append(embeds)
    return np.concatenate(all_embeds, axis=0)

In [ ]:
def apply_qe(embeddings, top_k=2):
    sim_matrix = embeddings @ embeddings.T
    indices = np.argsort(-sim_matrix, axis=1)[:, :top_k]
    expanded_embeddings = np.zeros_like(embeddings)
    for i in range(len(embeddings)):
        expanded_embeddings[i] = np.mean(embeddings[indices[i]], axis=0)
    return expanded_embeddings / np.linalg.norm(expanded_embeddings, axis=1, keepdims=True)

def k_reciprocal_rerank(embeddings, k1=6, k2=2, lambda_value=0.15):
    dist_matrix = 1 - (embeddings @ embeddings.T)
    original_dist = dist_matrix.copy()
    initial_rank = np.argsort(original_dist, axis=1)
    
    nn_k1 = []
    for i in range(embeddings.shape[0]):
        forward_k1 = initial_rank[i, :k1 + 1]
        backward_k1 = initial_rank[forward_k1, :k1 + 1]
        fi = np.where(backward_k1 == i)[0]
        nn_k1.append(forward_k1[fi])
        
    jaccard_dist = np.zeros_like(original_dist)
    for i in range(embeddings.shape[0]):
        temp_min = np.zeros(shape=[1, embeddings.shape[0]])
        ind_non_zero = np.where(original_dist[i, :] < 0.6)[0]
        ind_images = [inv for inv in ind_non_zero if len(np.intersect1d(nn_k1[i], nn_k1[inv])) > 0]
        for j in ind_images:
            intersection = len(np.intersect1d(nn_k1[i], nn_k1[j]))
            union = len(np.union1d(nn_k1[i], nn_k1[j]))
            jaccard_dist[i, j] = 1 - intersection / union
            
    return jaccard_dist * lambda_value + original_dist * (1 - lambda_value)

In [ ]:
init_embeds_matrix = tta_predict_embeds(model, processor, list(test_images.values()), aug_funcs[:2], batch_size=64)

In [ ]:
embeds_matrix = apply_qe(init_embeds_matrix, top_k=3)
name_to_idx = {name: i for i, name in enumerate(test_images.keys())}
dist_matrix = k_reciprocal_rerank(embeds_matrix)

In [ ]:
preds = []
for i in tqdm(range(len(test_data))):
    idx1 = name_to_idx[test_data['query_image'][i]]
    idx2 = name_to_idx[test_data['gallery_image'][i]]
    similarity = 1.0 - dist_matrix[idx1, idx2]
    preds.append(similarity)

min_val = min(preds)
max_val = max(preds)
preds = [(x - min_val) / (max_val - min_val) for x in preds]

submission = pd.DataFrame({
    'row_id': range(len(preds)),
    'similarity': preds
})
submission.to_csv('submission.csv', index=False)
submission.head()

In [ ]:
# preds = []
# for i in tqdm(range(len(test_data))):
#     embed1 = torch.tensor(embeds[test_data['query_image'][i]])
#     embed2 = torch.tensor(embeds[test_data['gallery_image'][i]])
#     similarity = F.cosine_similarity(embed1, embed2, dim=-1)
#     similarity = float(similarity)
#     preds.append(similarity)

# min_val = min(preds)
# max_val = max(preds)
# preds = [(x - min_val) / (max_val - min_val) for x in preds]

# submission = pd.DataFrame({
#     'row_id': range(len(preds)),
#     'similarity': preds
# })
# submission.to_csv('submission.csv', index=False)
# submission.head()